# Architectures: CNNs, RNNs, Transformers

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/T31SHA/data-ml-handbook/blob/main/notebooks/12_architectures.ipynb)

Runnable examples from **Data/ML Handbook** — section `12_architectures`. Read the explanations in the [docs](https://t31sha.github.io/data-ml-handbook/handbook/12_architectures/).

In [ ]:
# Setup: works in Colab or locally from the repo root
import os, subprocess, sys
if not os.path.exists("code") and not os.path.exists("../code"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/T31SHA/data-ml-handbook.git"], check=True)
    os.chdir("data-ml-handbook")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
if os.path.exists("../code"): os.chdir("..")
os.chdir("code"); os.makedirs("data", exist_ok=True)
for prep in ["proj01_make_csv.py", "s02_load.py"]:          # shared datasets used by many snippets
    if not os.path.exists("data/housing.parquet"): subprocess.run([sys.executable, prep], check=True)
sys.path.insert(0, "../figures/src")

### `code/s09_transfer.py`

In [ ]:
import torch, torch.nn as nn
from torchvision import models
m = models.resnet18(weights=None)                 # use weights="IMAGENET1K_V1" with internet access
for p in m.parameters(): p.requires_grad = False   # freeze backbone
m.fc = nn.Linear(m.fc.in_features, 5)              # new head (trainable)
train = sum(p.numel() for p in m.parameters() if p.requires_grad); total = sum(p.numel() for p in m.parameters())
print(f"trainable {train:,} / {total:,} params ({train/total:.2%})")
for p in m.layer4.parameters(): p.requires_grad = True       # stage 2: unfreeze last block
opt = torch.optim.AdamW([{"params": m.layer4.parameters(), "lr": 1e-4}, {"params": m.fc.parameters(), "lr": 1e-3}])
x = torch.randn(2, 3, 224, 224); print("logits shape:", tuple(m(x).shape))

### `code/s10_attention.py`

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
n, d, dk = 4, 8, 8
X = rng.normal(size=(n, d)); Wq, Wk, Wv = (rng.normal(size=(d, dk)) / np.sqrt(d) for _ in range(3))
Q, K, V = X @ Wq, X @ Wk, X @ Wv
def softmax(z): z = z - z.max(-1, keepdims=True); e = np.exp(z); return e / e.sum(-1, keepdims=True)
mask = np.triu(np.ones((n, n)), 1).astype(bool)                     # causal: no peeking at future tokens
S = Q @ K.T / np.sqrt(dk); S[mask] = -np.inf
A = softmax(S); out = A @ V
print("attention weights (rows sum to 1):\n", A.round(3)); print("output shape:", out.shape)
for dim in [16, 256, 4096]:                                         # why divide by sqrt(d)
    q, k = rng.normal(size=(10000, dim)), rng.normal(size=(10000, dim))
    print(f"d={dim:5d}  std(q·k)={np.std((q*k).sum(1)):7.2f}  std(q·k/√d)={np.std((q*k).sum(1)/np.sqrt(dim)):.2f}")